# 🏷️ Phân loại Tiêu đề Bài báo Công nghệ (IT vs Non-IT)

Fine-tune **PhoBERT** (`vinai/phobert-base`) để phân loại tiêu đề bài báo thành:
- **Label 0**: Tiêu đề KHÔNG liên quan đến IT
- **Label 1**: Tiêu đề liên quan đến IT

**Tech Stack**: PhoBERT + Underthesea + HuggingFace Transformers

**Môi trường**: Kaggle (GPU P100/T4)

## 1. Cài đặt packages

In [ ]:
!pip install -q underthesea transformers datasets accelerate scikit-learn

## 2. Import Libraries

In [ ]:
import json
import re
import numpy as np
import pandas as pd
from collections import Counter

from underthesea import word_tokenize, ner

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Kiểm tra GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 3. Load & Khám phá Dataset

In [ ]:
# === Cập nhật đường dẫn phù hợp với Kaggle ===
# Nếu upload file lên Kaggle dataset, đường dẫn sẽ dạng:
# DATA_PATH = '/kaggle/input/your-dataset-name/label_merged_titles.json'
# Nếu upload trực tiếp, dùng đường dẫn tương đối:
DATA_PATH = '/kaggle/input/label-merged-titles/label_merged_titles.json'

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)
print(f'Tổng số mẫu: {len(df)}')
print(f'\nPhân bố label:')
print(df['label'].value_counts())
print(f'\nTỷ lệ label 1 (IT): {df["label"].mean():.1%}')
print(f'\nCác nguồn tin:')
print(df['source_name'].value_counts())
print(f'\n--- Ví dụ tiêu đề Label 0 (Không IT) ---')
for t in df[df['label'] == 0]['title'].head(3).tolist():
    print(f'  • {t}')
print(f'\n--- Ví dụ tiêu đề Label 1 (IT) ---')
for t in df[df['label'] == 1]['title'].head(3).tolist():
    print(f'  • {t}')

## 4. Loại bỏ bản trùng lặp

In [ ]:
# Loại bỏ duplicate titles
original_count = len(df)
df = df.drop_duplicates(subset=['title'], keep='first').reset_index(drop=True)
print(f'Trước: {original_count} | Sau khi loại trùng: {len(df)} | Đã xóa: {original_count - len(df)}')
print(f'Phân bố label sau loại trùng:')
print(df['label'].value_counts())

## 5. Tiền xử lý với Underthesea (thay VnCoreNLP)

Pipeline tiền xử lý (tương tự hàm `del_test()` trong repo gốc):
1. **NER** → Thay thế tên riêng (ORG/PER → `name`, LOC → `loc`)
2. **Thay thế số liệu** → `percent`, `date`, `number`
3. **Loại bỏ dấu câu**
4. **Word Segmentation** bằng Underthesea
5. **Lowercase**

In [ ]:
def preprocess_title(text):
    """
    Tiền xử lý tiêu đề bài báo sử dụng Underthesea.
    Thay thế VnCoreNLP trong repo gốc.
    """
    if not text or not isinstance(text, str):
        return ''
    
    # 1. NER - Nhận diện thực thể bằng Underthesea
    try:
        entities = ner(text)
        # Thay thế từ cuối lên đầu để tránh lệch vị trí
        for word, pos, chunk, ner_tag in reversed(entities):
            if ner_tag in ('B-PER', 'I-PER', 'B-ORG', 'I-ORG'):
                text = text.replace(word, 'name', 1)
            elif ner_tag in ('B-LOC', 'I-LOC'):
                text = text.replace(word, 'loc', 1)
    except Exception:
        pass  # Nếu NER lỗi, bỏ qua bước này
    
    # 2. Thay thế phần trăm
    text = re.sub(r'\d+[.,]?\d*\s*%', 'percent', text)
    
    # 3. Thay thế ngày tháng (dd/mm/yyyy, mm/yyyy, etc.)
    text = re.sub(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}', 'date', text)
    text = re.sub(r'\d{1,2}[/-]\d{2,4}', 'date', text)
    
    # 4. Thay thế năm/tháng/quý
    text = re.sub(r'[Nn]ăm\s+\d{4}', 'date', text)
    text = re.sub(r'[Tt]háng\s+\d{1,2}', 'date', text)
    text = re.sub(r'[Qq]uý\s+\d', 'date', text)
    
    # 5. Thay thế số còn lại
    text = re.sub(r'\d+[.,]?\d*', 'number', text)
    
    # 6. Loại bỏ dấu câu
    text = re.sub(r'[.,;:!?\"\'\'\(\)\[\]\{\}\-–—…\"\"\'\'\`]', ' ', text)
    
    # 7. Word segmentation bằng Underthesea
    text = word_tokenize(text, format='text')  # Output: "khởi_nghiệp từ nấm sò"
    
    # 8. Lowercase + chuẩn hóa khoảng trắng
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    
    return text

In [ ]:
# Test thử preprocessing trên một vài mẫu
test_titles = [
    "Samsung xác nhận Galaxy S26 hỗ trợ AirDrop",
    "Hai kỹ sư FPT ghi dấu ấn tại hội nghị AI hàng đầu thế giới",
    "Cuộc thi AI cho học sinh toàn quốc, tổng giá trị giải thưởng 350 triệu đồng",
]
print('=== Test Preprocessing ===')
for title in test_titles:
    processed = preprocess_title(title)
    print(f'GỐC:  {title}')
    print(f'SAU:  {processed}')
    print()

In [ ]:
# Áp dụng preprocessing cho toàn bộ dataset
from tqdm import tqdm
tqdm.pandas()

print('Đang tiền xử lý toàn bộ dataset...')
df['processed_title'] = df['title'].progress_apply(preprocess_title)

# Kiểm tra kết quả
print(f'\nSố mẫu có processed_title rỗng: {(df["processed_title"] == "").sum()}')
print(f'\n--- Ví dụ trước/sau preprocessing ---')
for idx in [0, 1, 2]:
    print(f'GỐC:  {df.iloc[idx]["title"]}')
    print(f'SAU:  {df.iloc[idx]["processed_title"]}')
    print()

## 6. Chia dữ liệu: Train / Validation / Test

In [ ]:
# Chia 80% train, 10% val, 10% test (giống repo gốc)
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['label']
)

print(f'Train: {len(train_df)} mẫu')
print(f'  Label 0: {(train_df["label"] == 0).sum()} | Label 1: {(train_df["label"] == 1).sum()}')
print(f'Val:   {len(val_df)} mẫu')
print(f'  Label 0: {(val_df["label"] == 0).sum()} | Label 1: {(val_df["label"] == 1).sum()}')
print(f'Test:  {len(test_df)} mẫu')
print(f'  Label 0: {(test_df["label"] == 0).sum()} | Label 1: {(test_df["label"] == 1).sum()}')

## 7. Tokenization với PhoBERT

In [ ]:
MODEL_NAME = 'vinai/phobert-base'
MAX_LEN = 256  # PhoBERT max sequence length

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer loaded: {MODEL_NAME}')
print(f'Vocab size: {tokenizer.vocab_size}')

In [ ]:
# Tạo HuggingFace Datasets
def create_hf_dataset(dataframe):
    """Chuyển DataFrame thành HuggingFace Dataset đã tokenize."""
    dataset = Dataset.from_dict({
        'text': dataframe['processed_title'].tolist(),
        'label': dataframe['label'].tolist()
    })
    
    def tokenize_fn(examples):
        return tokenizer(
            examples['text'],
            padding='max_length',
            truncation=True,
            max_length=MAX_LEN
        )
    
    dataset = dataset.map(tokenize_fn, batched=True)
    dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    return dataset

train_dataset = create_hf_dataset(train_df)
val_dataset = create_hf_dataset(val_df)
test_dataset = create_hf_dataset(test_df)

print(f'Train dataset: {len(train_dataset)} samples')
print(f'Val dataset:   {len(val_dataset)} samples')
print(f'Test dataset:  {len(test_dataset)} samples')
print(f'\nSample features: {train_dataset[0].keys()}')
print(f'Input IDs shape: {train_dataset[0]["input_ids"].shape}')

## 8. Load PhoBERT Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'Non-IT', 1: 'IT'},
    label2id={'Non-IT': 0, 'IT': 1}
)

# Đếm tham số
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {MODEL_NAME}')
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## 9. Training Configuration & Train

In [ ]:
def compute_metrics(eval_pred):
    """Tính toán metrics cho Trainer."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    precision = precision_score(labels, predictions, average='macro')
    recall = recall_score(labels, predictions, average='macro')
    
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
OUTPUT_DIR = './phobert_title_classifier'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    
    # Epochs & Batch size
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    
    # Learning rate & Optimization
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    
    # Evaluation & Saving
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    
    # Performance
    fp16=True,  # Mixed precision trên GPU Kaggle
    
    # Logging
    logging_dir=f'{OUTPUT_DIR}/logs',
    logging_steps=10,
    report_to='none',  # Tắt wandb trên Kaggle
    
    # Misc
    seed=42,
    save_total_limit=2,  # Giữ tối đa 2 checkpoints
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('Training configuration:')
print(f'  Epochs: {training_args.num_train_epochs}')
print(f'  Batch size: {training_args.per_device_train_batch_size}')
print(f'  Learning rate: {training_args.learning_rate}')
print(f'  FP16: {training_args.fp16}')
print(f'  Best model metric: {training_args.metric_for_best_model}')

In [ ]:
# 🚀 Bắt đầu huấn luyện
print('=' * 60)
print('🚀 BẮT ĐẦU HUẤN LUYỆN')
print('=' * 60)

train_result = trainer.train()

print('\n' + '=' * 60)
print('✅ HUẤN LUYỆN HOÀN TẤT')
print('=' * 60)
print(f'Training loss: {train_result.training_loss:.4f}')
print(f'Training time: {train_result.metrics["train_runtime"]:.1f}s')

## 10. Đánh giá trên Test Set

In [ ]:
# Đánh giá trên test set
print('=' * 60)
print('📊 ĐÁNH GIÁ TRÊN TEST SET')
print('=' * 60)

test_results = trainer.evaluate(test_dataset)
print(f'\nTest Accuracy: {test_results["eval_accuracy"]:.4f}')
print(f'Test F1 Score: {test_results["eval_f1"]:.4f}')
print(f'Test Precision: {test_results["eval_precision"]:.4f}')
print(f'Test Recall:    {test_results["eval_recall"]:.4f}')

In [ ]:
# Classification Report & Confusion Matrix
test_predictions = trainer.predict(test_dataset)
y_pred = np.argmax(test_predictions.predictions, axis=-1)
y_true = test_predictions.label_ids

print('\n📋 Classification Report:')
print(classification_report(
    y_true, y_pred, 
    target_names=['Non-IT (0)', 'IT (1)']
))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, 
    display_labels=['Non-IT (0)', 'IT (1)']
)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Confusion Matrix - PhoBERT Title Classifier', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved to confusion_matrix.png')

## 11. Phân tích lỗi (Error Analysis)

In [ ]:
# Xem các mẫu bị dự đoán sai
test_df_copy = test_df.copy().reset_index(drop=True)
test_df_copy['predicted'] = y_pred
test_df_copy['correct'] = (y_pred == y_true)

wrong_predictions = test_df_copy[~test_df_copy['correct']]
print(f'Số mẫu dự đoán sai: {len(wrong_predictions)}/{len(test_df_copy)} ({len(wrong_predictions)/len(test_df_copy):.1%})')

if len(wrong_predictions) > 0:
    print(f'\n--- False Positives (Dự đoán IT nhưng thực tế Non-IT) ---')
    fp = wrong_predictions[(wrong_predictions['label'] == 0) & (wrong_predictions['predicted'] == 1)]
    for _, row in fp.head(5).iterrows():
        print(f'  • {row["title"]}')
    
    print(f'\n--- False Negatives (Dự đoán Non-IT nhưng thực tế IT) ---')
    fn = wrong_predictions[(wrong_predictions['label'] == 1) & (wrong_predictions['predicted'] == 0)]
    for _, row in fn.head(5).iterrows():
        print(f'  • {row["title"]}')

## 12. Hàm Inference (Dự đoán tiêu đề mới)

In [ ]:
def predict_title(title, model=model, tokenizer=tokenizer, show_detail=True):
    """
    Dự đoán label cho một tiêu đề mới.
    
    Args:
        title: Tiêu đề gốc (chưa preprocessing)
    Returns:
        dict: {label, label_name, confidence}
    """
    # Preprocessing
    processed = preprocess_title(title)
    
    # Tokenize
    inputs = tokenizer(
        processed,
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN,
        return_tensors='pt'
    ).to(device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        pred_label = torch.argmax(probs, dim=-1).item()
        confidence = probs[0][pred_label].item()
    
    label_name = 'IT' if pred_label == 1 else 'Non-IT'
    
    if show_detail:
        print(f'Tiêu đề: {title}')
        print(f'Đã xử lý: {processed}')
        print(f'Kết quả: {label_name} (label={pred_label}, confidence={confidence:.2%})')
        print(f'  P(Non-IT): {probs[0][0].item():.2%} | P(IT): {probs[0][1].item():.2%}')
        print()
    
    return {
        'label': pred_label,
        'label_name': label_name,
        'confidence': confidence,
        'prob_non_it': probs[0][0].item(),
        'prob_it': probs[0][1].item()
    }

In [ ]:
# Test dự đoán trên các tiêu đề mới
print('=' * 60)
print('🔮 TEST DỰ ĐOÁN TRÊN TIÊU ĐỀ MỚI')
print('=' * 60)

sample_titles = [
    # Nên là IT (label 1)
    'Google ra mắt mô hình AI Gemini 2.0 với khả năng lập trình tự động',
    'Hướng dẫn deploy ứng dụng React lên AWS Lambda',
    'Microsoft mua lại startup AI với giá 10 tỷ USD',
    
    # Nên là Non-IT (label 0)
    'Samsung ra mắt tủ lạnh thông minh mới tại Việt Nam',
    'Cách sạc pin điện thoại đúng cách để kéo dài tuổi thọ',
    'Tai nghe Sony WH-1000XM5 giảm giá sốc dịp Black Friday',
]

for title in sample_titles:
    predict_title(title)

## 13. Lưu Model

In [ ]:
# Lưu model tốt nhất
SAVE_PATH = './phobert_title_classifier_best'

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f'✅ Model đã được lưu tại: {SAVE_PATH}')
print(f'\nCác file đã lưu:')
import os
for f in os.listdir(SAVE_PATH):
    size = os.path.getsize(os.path.join(SAVE_PATH, f))
    print(f'  {f} ({size/1024/1024:.1f} MB)' if size > 1024*1024 else f'  {f} ({size/1024:.1f} KB)')

In [ ]:
# Cách load lại model đã lưu
print('=== Cách load lại model ===')
print('''
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load model
tokenizer = AutoTokenizer.from_pretrained('./phobert_title_classifier_best')
model = AutoModelForSequenceClassification.from_pretrained('./phobert_title_classifier_best')
model.eval()

# Dùng hàm predict_title() để dự đoán
''')

## 14. Training History

In [ ]:
# Vẽ biểu đồ training history
log_history = trainer.state.log_history

# Lấy eval metrics theo epoch
eval_logs = [log for log in log_history if 'eval_loss' in log]
train_logs = [log for log in log_history if 'loss' in log and 'eval_loss' not in log]

if eval_logs:
    epochs = list(range(1, len(eval_logs) + 1))
    eval_acc = [log.get('eval_accuracy', 0) for log in eval_logs]
    eval_f1 = [log.get('eval_f1', 0) for log in eval_logs]
    eval_loss = [log.get('eval_loss', 0) for log in eval_logs]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Accuracy
    axes[0].plot(epochs, eval_acc, 'o-', color='#2196F3', linewidth=2, markersize=8)
    axes[0].set_title('Validation Accuracy', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim([0.5, 1.0])
    
    # F1 Score
    axes[1].plot(epochs, eval_f1, 's-', color='#4CAF50', linewidth=2, markersize=8)
    axes[1].set_title('Validation F1 Score', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('F1 Score')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([0.5, 1.0])
    
    # Loss
    axes[2].plot(epochs, eval_loss, 'D-', color='#F44336', linewidth=2, markersize=8)
    axes[2].set_title('Validation Loss', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Loss')
    axes[2].grid(True, alpha=0.3)
    
    plt.suptitle('PhoBERT Title Classifier - Training History', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Training history saved to training_history.png')
else:
    print('Không có eval logs')

---
## Tóm tắt

| Thành phần | Chi tiết |
|------------|----------|
| **Model** | `vinai/phobert-base` (RoBERTa architecture) |
| **NLP Preprocessing** | Underthesea (thay VnCoreNLP) |
| **Task** | Binary classification (Non-IT vs IT) |
| **Dataset** | 1020 tiêu đề bài báo công nghệ |
| **Split** | 80% train / 10% val / 10% test |
| **Optimizer** | AdamW (lr=2e-5, weight_decay=0.01) |
| **Epochs** | 5 (with early stopping patience=2) |